In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)
os.environ.pop("HADOOP_CONF_DIR", None)

'/usr/local/hadoop/etc/hadoop'

In [2]:
import sys, os, pyspark
print("python:", sys.executable)
print("pyspark:", pyspark.__file__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_CONF_DIR:", os.environ.get("HADOOP_CONF_DIR"))

python: /home/hadoop/project-de-zoomcamp/.venv/bin/python
pyspark: /home/hadoop/project-de-zoomcamp/.venv/lib/python3.12/site-packages/pyspark/__init__.py
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: None
PYSPARK_PYTHON: None
HADOOP_CONF_DIR: None


In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [4]:
pyspark.__version__

'4.1.1'

In [5]:
credentials_location = '/home/hadoop/project-de-zoomcamp/terraform/keys/de-zoomcamp-493207-d3f7354bdd98.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "/home/hadoop/project-de-zoomcamp/lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [6]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/04/17 15:03:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [7]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [8]:
df = spark.read.parquet(
    "gs://de-zoomcamp-493207-healthcare-lake/healthcare_vitals/event_date=2026-04-17/*"
)
df.show()

26/04/17 15:03:32 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://de-zoomcamp-493207-healthcare-lake/healthcare_vitals/event_date=2026-04-17/*.
java.io.FileNotFoundException: File not found: gs://de-zoomcamp-493207-healthcare-lake/healthcare_vitals/event_date=2026-04-17/*
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getFileStatus(GoogleHadoopFileSystemBase.java:958)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.$anonfun$applyOrElse$2(ResolveDataSource.scala:61)
	at scala.Option.getOrElse(Option.scala:201)
	at org.ap

+----------+---------+--------------------+----------+------------+-----------+------------+----------------+--------------+----------+
|subject_id|device_id|          event_time|heart_rate|oxygen_level|systolic_bp|diastolic_bp|body_temperature|activity_level|alert_flag|
+----------+---------+--------------------+----------+------------+-----------+------------+----------------+--------------+----------+
|     17586|    10009|2026-04-17 14:59:...|        60|          97|        104|          73|            36.8|      sleeping|     false|
|     33064|    10021|2026-04-17 14:59:...|       107|          96|        125|          82|            37.0|       running|     false|
|     38251|    10021|2026-04-17 14:59:...|        72|          96|        105|          73|            36.6|       resting|     false|
|     13894|    10017|2026-04-17 14:59:...|        80|          96|        107|          73|            36.8|       resting|     false|
|     42971|    10003|2026-04-17 14:59:...|     